# CatBoost only for cat cols

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split

In [4]:
# File Paths for train and test data
BASE_PATH = r"../playground-series-s5e12/"
TRAIN_PATH = BASE_PATH + "train.csv"
TEST_PATH =  BASE_PATH + "test.csv"

In [5]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [6]:
NUM_COLS = [
    'age', 
    'alcohol_consumption_per_week', 
    'physical_activity_minutes_per_week', 
    'diet_score', 
    'sleep_hours_per_day', 
    'screen_time_hours_per_day', 
    'bmi', 
    'waist_to_hip_ratio', 
    'systolic_bp', 
    'diastolic_bp', 
    'heart_rate', 
    'cholesterol_total', 
    'hdl_cholesterol', 
    'ldl_cholesterol', 
    'triglycerides', 
]

CAT_COLS = [
    'gender',
    'ethnicity', 
    'education_level', 
    'income_level', 
    'smoking_status', 
    'employment_status',
    'family_history_diabetes', # Binary 1/0
    'hypertension_history', # Binary 1/0
    'cardiovascular_history', # Binary 1/0
    # 'diagnosed_diabetes' # Binary 1/0
]

TARGET = 'diagnosed_diabetes'


In [10]:
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import numpy as np


df = train.copy()
df_test = test.copy()
# X = features, y = target column
X = df.drop(["id", TARGET], axis=1)
y = df[TARGET]
X_pred_id = df_test["id"]
X_pred = df_test.drop("id", axis=1)

# Auto-detect all categorical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
accuracies = []
f1_scores = []
roc = []

test_preds = np.zeros(len(X_pred))
predsonly = []

for train_idx, test_idx in skf.split(X, y):
    print(f"\n===== Fold {fold} =====")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # CatBoost Pool
    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    test_pool = Pool(X_test, y_test, cat_features=cat_cols)
    pred_pool = Pool(X_pred, cat_features=cat_cols)


    # Class imbalance handling with auto_class_weights
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        auto_class_weights="Balanced",   # <-- handles imbalance
        verbose=False
    )

    # Train on this fold
    model.fit(train_pool)

    # Predict
    y_pred = model.predict(test_pool)
    y_pred_proba = model.predict_proba(test_pool)[:,1]
    # test_preds += model.predict_proba(pred_pool)[:,1]
    predsonly.append(model.predict_proba(pred_pool))
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_pred_proba)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    print(f"ROC AUC Score: {roc}")
    print(f"Accuracy: {acc:.4f}, F1-score: {f1:.4f}")
    print(classification_report(y_test, y_pred))

    fold += 1

print("\n===== Final Cross-Validation Results =====")
print("Mean Accuracy:", np.mean(accuracies))
print("Mean F1 Score:", np.mean(f1_scores))
print("Mean Roc Auc Score:", np.mean(roc))



===== Fold 1 =====
ROC AUC Score: 0.7213911230310583
Accuracy: 0.6495, F1-score: 0.6875
              precision    recall  f1-score   support

         0.0       0.53      0.70      0.60     52738
         1.0       0.77      0.62      0.69     87262

    accuracy                           0.65    140000
   macro avg       0.65      0.66      0.64    140000
weighted avg       0.68      0.65      0.65    140000


===== Fold 2 =====
ROC AUC Score: 0.7197462643646781
Accuracy: 0.6479, F1-score: 0.6852
              precision    recall  f1-score   support

         0.0       0.52      0.70      0.60     52738
         1.0       0.77      0.61      0.69     87262

    accuracy                           0.65    140000
   macro avg       0.65      0.66      0.64    140000
weighted avg       0.68      0.65      0.65    140000


===== Fold 3 =====
ROC AUC Score: 0.7202931649613005
Accuracy: 0.6477, F1-score: 0.6856
              precision    recall  f1-score   support

         0.0       0.52 

In [11]:
predsonly

[array([[0.59890474, 0.40109526],
        [0.46040542, 0.53959458],
        [0.32968356, 0.67031644],
        ...,
        [0.61317009, 0.38682991],
        [0.50626243, 0.49373757],
        [0.5306967 , 0.4693033 ]], shape=(300000, 2)),
 array([[0.59731184, 0.40268816],
        [0.46335552, 0.53664448],
        [0.34887464, 0.65112536],
        ...,
        [0.58785079, 0.41214921],
        [0.50219621, 0.49780379],
        [0.52817633, 0.47182367]], shape=(300000, 2)),
 array([[0.59877824, 0.40122176],
        [0.45109719, 0.54890281],
        [0.34655522, 0.65344478],
        ...,
        [0.62561923, 0.37438077],
        [0.50418769, 0.49581231],
        [0.53405644, 0.46594356]], shape=(300000, 2)),
 array([[0.5948706 , 0.4051294 ],
        [0.4649672 , 0.5350328 ],
        [0.33909126, 0.66090874],
        ...,
        [0.60710632, 0.39289368],
        [0.52443444, 0.47556556],
        [0.53579705, 0.46420295]], shape=(300000, 2)),
 array([[0.59781061, 0.40218939],
        [0.453

In [14]:
pred_arr = np.zeros(300000)

In [15]:
for i in predsonly:
    pred_arr+= i[:,1]/5

In [12]:
test.shape

(300000, 25)

In [17]:
pd.DataFrame({"id":X_pred_id, TARGET:pred_arr}).to_csv("catboostsub01.csv", index = False)

In [18]:
# 0.69586 Public Score